In [0]:
import pandas as pd
import numpy as np

In [0]:
df = pd.read_csv(r'/Volumes/workspace/default/agentes_ia/SMS_train_sintetico.csv', encoding='cp1252', sep=',')
df_teste = pd.read_csv(r'/Volumes/workspace/default/agentes_ia/SMS_test_sintetico.csv', encoding='cp1252', sep=',')
print(df.sample(2))
print("\n===================\n")
print(df_teste.sample(2))

# O enconding antes estava como 'ANSI', onde, no Windows, “ANSI” é um apelido usado pelo Bloco de Notas e pelo Excel para a code page padrão do sistema, que em português/Brasil é a Windows-1252 (também chamada cp1252). O Python não conhece esse apelido, então dá LookupError. O equivalente a este enconding é 'cp1252' ou 'latin-1' com replace.

# O r a frente do caminho serve para dizer ao Python para ignorar caracteres de escape (como \n, \t ou \b) e tratar todas as barras invertidas (\) como texto comum.

In [0]:
df['Label'].value_counts()

In [0]:
df['Label'].unique()

Criar um TF-IDF com n-grams de 1 a N

In [0]:
 %pip install nltk

In [0]:
from nltk.downloader import download # O NLTK (Natural Language Toolkit) não vem com seus corpora e listas de palavras embutidos.
download('stopwords')

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

In [0]:
# Definindo as stopwords em inglẽs
stop_words_english = list(stopwords.words('english'))

In [0]:
# Criação do vetor TF-IDF
Tfidf = TfidfVectorizer(ngram_range=(1, 4),
                      stop_words=stop_words_english, #Já remove as stopwords
                      strip_accents = 'unicode', #Já remove os acentos do texto
                      max_features=512, #Pega as n palavras mais frequentes e corta o número de features
                      norm = 'l1' 
                      #l1 soma do vetor de cada frase é 1;
                      #l2 soma do quadrado dos elementos do vetor é 1; None sem normalizar
                        )
# Corpus
corpus = df['Message_body'].values

In [0]:
# Transformação das corpus em matriz TF-IDF e convertendo para array
Tfidf_X = Tfidf.fit_transform(corpus).toarray()

In [0]:
# Salvar no dataframe

df['vetores_tfidf'] = list(Tfidf_X)
df.head()

In [0]:
df_teste['vetores_tfidf'] = list(Tfidf.transform(df_teste['Message_body'].values).toarray())
df_teste.head()

Criar um modelo de classificação supervisionado com base na coluna target

In [0]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

In [0]:
# Preparar dados - Converter para numpy array 2D
X = np.array(df['vetores_tfidf'].tolist())
y = df['Label'].values

In [0]:
# Label Encoder para converter labels em números
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [0]:
# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [0]:
# Treinar modelo de Regressão Logística
print("Treinando modelo de Regressão Logística...")
model_lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
model_lr.fit(X_train, y_train)

In [0]:
# Predições
y_train_pred = model_lr.predict(X_train)
y_test_pred = model_lr.predict(X_test)

In [0]:
# Scores
train_score = model_lr.score(X_train, y_train)
test_score = model_lr.score(X_test, y_test)

In [0]:
# Classification Report para Treino
print("\n" + "="*60)
print("CLASSIFICATION REPORT - TREINO")
print("="*60)
print(classification_report(y_train, y_train_pred, target_names=le.classes_))

# Classification Report para Teste
print("\n" + "="*60)
print("CLASSIFICATION REPORT - TESTE")
print("="*60)
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

In [0]:
# Realizar inferencia para dados novos
X_inf = np.array(df_teste['vetores_tfidf'].tolist())
df_teste['classe'] = le.inverse_transform(model_lr.predict(X_inf))
df_teste

In [0]:
# Classification Report para novos dados
print("\n" + "="*60)
print("CLASSIFICATION REPORT - VALIDAÇÃO")
print("="*60)
print(classification_report(df_teste['Label'], df_teste['classe']))